Importing packages

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import joblib


Loading Dataset

In [22]:
df = pd.read_csv('Crop_recommendation.csv')

print(df.head())

    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice


Defining x and y features

In [23]:
X = df.drop('label', axis=1)
y = df['label']


Splittibng x and y into training and testing

In [24]:


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

y_train_cat = to_categorical(y_train_enc)
y_test_cat = to_categorical(y_test_enc)


Creating Model

In [25]:

model = Sequential()
model.add(Dense(128, input_shape=(X_train.shape[1],), activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dense(22, activation='softmax'))  

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


C:\Users\Manish Kumar Jain\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training model

In [26]:

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    shuffle=True
)


Epoch 1/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.0860 - loss: 2.9786 - val_accuracy: 0.4636 - val_loss: 2.4220
Epoch 2/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3991 - loss: 2.2584 - val_accuracy: 0.7109 - val_loss: 1.2966
Epoch 3/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6242 - loss: 1.2894 - val_accuracy: 0.8145 - val_loss: 0.6948
Epoch 4/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7255 - loss: 0.8521 - val_accuracy: 0.8745 - val_loss: 0.4737
Epoch 5/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7597 - loss: 0.6772 - val_accuracy: 0.9200 - val_loss: 0.3526
Epoch 6/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8203 - loss: 0.5489 - val_accuracy: 0.9182 - val_loss: 0.3045
Epoch 7/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8264 - loss: 0.5011 - val_accuracy: 0.9218 - val_loss: 0.2624
Epoch 8/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8269 - loss: 0.4623 - val_accuracy: 0.9327 - v

Checking trained model

In [27]:
label_encoder = LabelEncoder()
label_encoder.fit(y) 

samples = np.array([
    [90, 42, 43, 20.5, 80.0, 6.5, 120],
    [50, 35, 45, 22.0, 60.0, 6.0, 100],
    [10, 5, 0, 25.0, 85.0, 5.5, 250],
    [80, 60, 30, 27.5, 70.0, 6.8, 200],
    [100, 80, 40, 15.0, 55.0, 5.0, 50],
    [85, 58, 40, 30.0, 75.0, 6.2, 110],
    [60, 25, 50, 24.0, 68.0, 6.1, 90],
    [20, 10, 15, 19.0, 90.0, 6.3, 180],
    [40, 40, 50, 21.0, 65.0, 6.5, 80],
    [25, 30, 40, 23.0, 72.0, 6.4, 130],
])

samples_scaled = scaler.transform(samples)


predictions = model.predict(samples_scaled)


for idx, probs in enumerate(predictions):
    print(f"\n🔹 Sample {idx+1}: {samples[idx].tolist()}")
    
    top_3_indices = probs.argsort()[-3:][::-1]
    
    for rank, i in enumerate(top_3_indices, 1):
        crop = label_encoder.inverse_transform([i])[0]
        confidence = probs[i] * 100
        print(f"   ➔ {crop}: {confidence:.2f}% confidence")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step


C:\Users\Manish Kumar Jain\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



🔹 Sample 1: [90.0, 42.0, 43.0, 20.5, 80.0, 6.5, 120.0]
   ➔ jute: 43.40% confidence
   ➔ cotton: 31.97% confidence
   ➔ coffee: 17.92% confidence

🔹 Sample 2: [50.0, 35.0, 45.0, 22.0, 60.0, 6.0, 100.0]
   ➔ coffee: 75.38% confidence
   ➔ watermelon: 10.89% confidence
   ➔ maize: 6.98% confidence

🔹 Sample 3: [10.0, 5.0, 0.0, 25.0, 85.0, 5.5, 250.0]
   ➔ coconut: 99.96% confidence
   ➔ rice: 0.03% confidence
   ➔ orange: 0.01% confidence

🔹 Sample 4: [80.0, 60.0, 30.0, 27.5, 70.0, 6.8, 200.0]
   ➔ jute: 51.19% confidence
   ➔ rice: 48.42% confidence
   ➔ coffee: 0.23% confidence

🔹 Sample 5: [100.0, 80.0, 40.0, 15.0, 55.0, 5.0, 50.0]
   ➔ maize: 99.92% confidence
   ➔ kidneybeans: 0.05% confidence
   ➔ cotton: 0.03% confidence

🔹 Sample 6: [85.0, 58.0, 40.0, 30.0, 75.0, 6.2, 110.0]
   ➔ banana: 95.08% confidence
   ➔ jute: 2.51% confidence
   ➔ cotton: 1.52% confidence

🔹 Sample 7: [60.0, 25.0, 50.0, 24.0, 68.0, 6.1, 90.0]
   ➔ watermelon: 94.24% confidence
   ➔ coffee: 4.10% confidenc

Saving model

In [28]:


# 1. Save the trained model
model.save('crop_recommendation_model.h5')  # saves in HDF5 format

# 2. Save the scaler
joblib.dump(scaler, 'crop_recommendation_scaler.pkl')

# 3. Save the label encoder
joblib.dump(label_encoder, 'crop_recommendation_label_encoder.pkl')

print("✅ Model, scaler, and label encoder saved successfully!")


✅ Model, scaler, and label encoder saved successfully!
